In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)
import importlib
import os
import sys
#root_path = os.path.dirname(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)
from env.parameters import P
from analysis_functions.data_preparation import cohort_type_adjustment
import dask.dataframe as dd
import pickle
import yaml
from analysis_functions.feature_engineering import (
keep_england_country_imd,
drop_unknown_country_imd,
imd_quantiles,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
find_normal_boundaries, 
find_skewed_boundaries,
diagnostic_plots,
plot_boxplot_and_hist,
outlier_analysis,
keep_england_country_imd,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
impute_nulls_mice,
create_age_bands
)

from analysis_functions.custom_transformers import (
Custom_Winsoriser,
bmi_categoriser,
fev1fvc_ratio_categoriser,
traffic_intensity_quantiles,
inverse_distance_quantiles,
CustomFrequencyBinner,
CustomWaistBinner,
MultiTransform,
MultiTransformList,
CustomBMICategoriser,
CustomFev1FvcRatioCategoriser,
CustomInverseDistanceCategoriser,
CustomTrafficIntensityCategoriser,
ColumnSelector,
CustomBinaryCategoriserAroundMean,
CustomBinaryCategoriserAroundMedian,
CustomBinaryCategoriserAroundDecile
)
from scipy.stats.mstats import winsorize
from fancyimpute import IterativeImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import warnings
from diffprivlib.utils import PrivacyLeakWarning
from sklearn.decomposition import PCA
import diffprivlib as dp
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, confusion_matrix, 
classification_report, f1_score, 
roc_curve, roc_auc_score,
precision_recall_curve,
precision_score, recall_score, average_precision_score,balanced_accuracy_score, matthews_corrcoef)
import shap
from scipy.stats import chi2
from collections import Counter


In [ ]:
cohort_path = P.cohorts_ukb_start_gphesonly_path

In [ ]:
df_in = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/epi_analysis_ready_df_ukb_start_gphesonly.csv''')

In [ ]:
pickle_file = f'''{P.cohorts_ukb_start_gphesonly_path}/pickle/cols_dict_gphesonly_all.pickle'''

with open(pickle_file, 'rb') as f:
     cols_dict = pickle.load(f)

print(cols_dict.keys())

In [ ]:
print(df_in.shape)



In [ ]:
# Adjust feature types
df_in = cohort_type_adjustment(df_in, cols_dict)

In [ ]:
from pipeline_functions import *
from custom_plots import *
from epi_functions import *

In [ ]:
df_in["country_imd"].value_counts(dropna=False)

In [ ]:
df_in.shape

In [ ]:
from sklearn.neighbors import NearestNeighbors
from scipy.stats import fisher_exact, norm, chi2_contingency
from statsmodels.stats.contingency_tables import Table2x2


# How many with less than 1 year of follow up?

In [ ]:
col_o = "flag_post_cohort_start_exac_ocs_y1"

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1].shape[0]

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1]["follow_up_asthma_pre_cohort_start"].describe()

In [ ]:
# Drop these
df = df_in[df_in["follow_up_asthma_pre_cohort_start"]>=1]

In [ ]:
df["follow_up_asthma_pre_cohort_start"].describe()

In [ ]:
df["follow_up_asthma_pre_cohort_start"].hist()

In [ ]:
df.loc[:, "follow_up_asthma_over_12"] = df["follow_up_asthma_pre_cohort_start"].apply(lambda x: 1 if x >=12 else 0)

In [ ]:
df["follow_up_asthma_over_12"].value_counts()

In [ ]:
df["age_asthma"].hist()

In [ ]:
df.loc[:, "late_onset_asthma_40"] = df["age_asthma"].apply(lambda x: 1 if x >=40 else 0)

In [ ]:
df["late_onset_asthma_40"].value_counts()

In [ ]:
# Any outcome on study start?

df[df['evdt_first_post_cohort_start_exac']==df['evdt_cohort_start']]

In [ ]:
print(df_in.shape)
print(df.shape)

In [ ]:
df_in.shape[0] - df.shape[0]

In [ ]:
df[col_o].value_counts()/df.shape[0]*100

# Cardinal symptoms

In [ ]:
df.loc[:, "cardinal_symptoms"] =df[["wheeze_field", "shortness_breath_field", "chest_pain_field"]].any(axis=1).astype(int)

In [ ]:
df["cardinal_symptoms"].value_counts()

# How many deaths before the outcome (do not censor). Just do sensitivity without these later

In [ ]:
df[["evdt_cohort_start","evdt_first_post_cohort_start_exac", "dod"]].dtypes

In [ ]:
#condition = (df[col_o]==1) & (df['dod'].notnull()) & ((df['dod'] >= df['evdt_first_post_cohort_start_exac']) & (df['dod'] <= df['evdt_first_post_cohort_start_exac'] + pd.Timedelta(days=365)))
condition = (df['dod'].notnull()) & (df['dod'] > df['evdt_cohort_start']) & (df['dod'] <= df['evdt_cohort_start'] + pd.Timedelta(days=365))

df[condition].shape

In [ ]:
# any exac in these?
df[condition & df[col_o]==1][["evdt_cohort_start", "evdt_first_post_cohort_start_exac", "dod"]].shape

In [ ]:
df['tte_cohort_start_to_exac'].describe()

In [ ]:
# any deaths recorded before exac 1 event data? any deaths in non-exac within a year?

df[df['dod']<df['evdt_first_post_cohort_start_exac']]

# OHE Smoking and Ethnicity

- We have not dropped any level here. Drop in modeling

In [ ]:
df = pd.get_dummies(df, columns=['eth_grouped_1b', 'desc_smoking_at_baseline'], drop_first=False)
dummy_columns = [col for col in df.columns if 'eth_grouped_1b_' in col or 'desc_smoking_at_baseline_' in col]
df[dummy_columns] = df[dummy_columns].astype(int)

In [ ]:
df['Smoker_current'] = df["desc_smoking_at_baseline_Current"].apply(lambda x: 1 if x ==1 else 0)

# Full model

In [ ]:
df.columns

In [ ]:
# Covariates
cov_list= ['age_60+', 'sex_female', 
           'eth_non_white',
           'late_onset_asthma_40',
            'pheno_anxiety_pre_cohort_start',
           'bmi_30_imputed', 
           'pheno_ckd_pre_cohort_start',
           'pheno_copd_pre_cohort_start',
           'pheno_cvd_pre_cohort_start',
            'pheno_depression_pre_cohort_start',
           'pheno_diabetes_pre_cohort_start',
           'pheno_ht_pre_cohort_start',
           'cardinal_symptoms',
            'flag_pre_cohort_start_exac_y1', 'flag_pre_cohort_start_meds_ocs_y1',
            'flag_pre_cohort_start_meds_other_y1',
'Smoker_current',
          'major_road_field'] 



In [ ]:
cov_list

In [ ]:
rename_dict = {
 'age_60+': "Age60+",
 'sex_female': "Sex_female",
 'eth_non_white': "Non_white",
 'late_onset_asthma_40': "Late_onset",
 'pheno_anxiety_pre_cohort_start': "Anxiety",
 'bmi_30_imputed': "BMI>30",
 'pheno_ckd_pre_cohort_start': "CKD",
 'pheno_copd_pre_cohort_start' :"COPD",
 'pheno_cvd_pre_cohort_start': "CVD",
 'pheno_depression_pre_cohort_start': "Depression",
 'pheno_diabetes_pre_cohort_start': "Diabetes",
 'pheno_ht_pre_cohort_start' : "Hypertension",
 'cardinal_symptoms': "Cardinal_symptomps",
 'flag_pre_cohort_start_exac_y1':  "Pre_baseline_exacerbation",
 'flag_pre_cohort_start_meds_ocs_y1': "Pre_baseline_OCS",
 'flag_pre_cohort_start_meds_other_y1': "Pre_baseline_meds",
'Smoker_current': 'Smoker_current',
 'major_road_field': "Near_major_road"
}

In [ ]:
df = df.rename(columns=rename_dict)

In [ ]:
cov_list_full = rename_dict.values()

In [ ]:
cov_list_full

In [ ]:
df[cov_list_full].head()

In [ ]:
X_full = df[cov_list_full]
y = df[[col_o]].values.flatten()





In [ ]:
# Train the full model
model_full = LogisticRegression(random_state=7)
model_full.fit(X_full, y)

# Near full

In [ ]:
cov_list_near_full = ['Age60+', 'Sex_female', 'Non_white', 'Late_onset', 'Anxiety', 'BMI>30', 'CKD', 'COPD', 'CVD', 'Depression', 'Diabetes', 'Hypertension', 'Cardinal_symptomps', 'Pre_baseline_exacerbation', 'Pre_baseline_OCS', 'Pre_baseline_meds', 'Smoker_current']

In [ ]:
cov_list_near_full

In [ ]:
X_near_full = df[cov_list_near_full]

In [ ]:
model_near_full = LogisticRegression(random_state=7)
model_near_full.fit(X_near_full, y)

# Partial model

In [ ]:
cov_list_partial = ['Age60+', 'Sex_female', 'Non_white', 'Late_onset', 'Anxiety', 'BMI>30', 'CKD', 'COPD', 'CVD', 'Depression', 'Diabetes', 'Hypertension', 'Cardinal_symptomps', 'Pre_baseline_exacerbation', 'Pre_baseline_OCS', 'Smoker_current']

In [ ]:
cov_list_partial

In [ ]:
X_partial = df[cov_list_partial]

In [ ]:
model_partial = LogisticRegression(random_state=7)
model_partial.fit(X_partial, y)

# Simplest model

In [ ]:
cov_list_simple=['Age60+', 'Sex_female', 'Non_white', 'Anxiety', 'BMI>30',  'COPD', 'Diabetes', 'Hypertension', 'Cardinal_symptomps', 'Pre_baseline_exacerbation', 'Pre_baseline_OCS']

In [ ]:
cov_list_simple

In [ ]:
X_simple = df[cov_list_simple]

In [ ]:
model_simple = LogisticRegression(random_state=7)
model_simple.fit(X_simple, y)

# Basic model

In [ ]:
cov_list_basic=['Age60+', 'Sex_female', 'Anxiety', 'BMI>30', 'COPD', 'Diabetes', 'Hypertension', 'Cardinal_symptomps', 'Pre_baseline_exacerbation', 'Pre_baseline_OCS']

In [ ]:
cov_list_basic

In [ ]:
X_basic = df[cov_list_basic]


In [ ]:
model_basic = LogisticRegression(random_state=7)
model_basic.fit(X_basic, y)

In [ ]:
#minimum
cov_list_min=['Age60+', 'Sex_female',  'BMI>30', 'COPD',  'Cardinal_symptomps']
X_minimum = df[cov_list_min]
model_minimum = LogisticRegression(random_state=7)
model_minimum.fit(X_minimum, y)


# Log likelihood test

In [ ]:
def log_likelihood_test(model_full, X_full, model_simple, X_simple, y):
    """
    Compares two logistic regression models (nested) using the Likelihood Ratio Test.
    
    Parameters:
    model_full : fitted logistic regression model (full model with more parameters)
    X_full     : feature matrix used for the full model
    model_simple : fitted logistic regression model (simpler model with fewer parameters)
    X_simple     : feature matrix used for the simpler model
    y           : true binary labels (0 or 1)
    
    Returns:
    A dictionary with the likelihood ratio statistic, degrees of freedom, and p-value.
    """
    def log_likelihood(model, X, y):
        probs = model.predict_proba(X)[:, 1]
        ll = y * np.log(probs) + (1 - y) * np.log(1 - probs)
        return ll.sum()
    log_likelihood_full = log_likelihood(model_full, X_full, y)
    log_likelihood_simple = log_likelihood(model_simple, X_simple, y)
    lr_stat = 2 * (log_likelihood_full - log_likelihood_simple)
    df_p = X_full.shape[1] - X_simple.shape[1]
    p_value = chi2.sf(lr_stat, df_p)
    return {
        'Likelihood Ratio Statistic': lr_stat,
        'Degrees of Freedom': df_p,
        'P-value': p_value
    }

In [ ]:
log_likelihood_test(model_full=model_full, 
                    X_full=X_full, 
                    model_simple=model_near_full, 
                    X_simple=X_near_full, y= y)

In [ ]:
log_likelihood_test(model_full=model_near_full, 
                    X_full=X_near_full, 
                    model_simple=model_partial, 
                    X_simple=X_partial, y= y)

null hypothesis:simpler model (the null model) is sufficient to explain the data, and the additional predictors in the full model do not significantly improve the fit.

p<0.05: This indicates that the full model provides a significantly better fit to the data than the null model.

In [ ]:
log_likelihood_test(model_full=model_partial, 
                    X_full=X_partial, 
                    model_simple=model_simple, 
                    X_simple=X_simple, y= y)

In [ ]:
log_likelihood_test(model_full=model_simple, 
                    X_full=X_simple, 
                    model_simple=model_basic, 
                    X_simple=X_basic, y= y)

In [ ]:
log_likelihood_test(model_full=model_basic, 
                    X_full=X_basic, 
                    model_simple=model_minimum, 
                    X_simple=X_minimum, y= y)